# Lecturas consolidadas

Carga de tablas crudas y consolidación de `measurements` con `sensor_types` y `device_nodes`.

In [ ]:
from pathlib import Path

import pandas as pd


In [ ]:
DATA_DIR = Path('../data/raw')

MEASUREMENTS_PATH = DATA_DIR / 'measurements.csv'
SENSOR_TYPES_PATH = DATA_DIR / 'sensor_types.csv'
NODES_DEVICES_PATH = DATA_DIR / 'device_nodes.csv'

COMMON_AUDIT_COLUMNS = ['created_at', 'updated_at', 'created_by', 'updated_by']
MEASUREMENTS_DROP_COLUMNS = [*COMMON_AUDIT_COLUMNS, 'measured_at']

DISPLAY_COLUMNS = [
    'measurement_id',
    'timestamp',
    'node_id',
    'model',
    'sensor_type_id',
    'name',
    'value',
    'unit_of_measure',
]


In [ ]:
def load_csv(path: Path, drop_columns: list[str] | None = None, parse_dates: list[str] | None = None) -> pd.DataFrame:
    """Lee un CSV y elimina columnas solo si existen."""
    dataframe = pd.read_csv(path, parse_dates=parse_dates)

    if drop_columns:
        existing_columns = [column for column in drop_columns if column in dataframe.columns]
        dataframe = dataframe.drop(columns=existing_columns)

    return dataframe


In [ ]:
measurements_df = load_csv(
    MEASUREMENTS_PATH,
    drop_columns=MEASUREMENTS_DROP_COLUMNS,
    parse_dates=['timestamp'],
)

measurements_df.head()


In [ ]:
sensor_types_df = load_csv(SENSOR_TYPES_PATH, drop_columns=COMMON_AUDIT_COLUMNS)
sensor_types_df.head()


In [ ]:
nodes_devices_df = load_csv(
    NODES_DEVICES_PATH,
    drop_columns=COMMON_AUDIT_COLUMNS,
    parse_dates=['activated_at'],
)

nodes_devices_df.head()


In [ ]:
consolidado_df = (
    measurements_df.merge(
        sensor_types_df,
        on='sensor_type_id',
        how='left',
        validate='many_to_one',
        suffixes=('', '_sensor'),
    )
    .merge(
        nodes_devices_df,
        on='node_id',
        how='left',
        validate='many_to_one',
        suffixes=('', '_node'),
    )
    .sort_values(['timestamp', 'node_id', 'sensor_type_id'])
    .reset_index(drop=True)
)

consolidado_df[DISPLAY_COLUMNS].head()


In [ ]:
consolidado_df.info()
